# 🛡️ Glu-Stock: 03_EXECUTION_ENGINE
**Phase**: Risk Management & Trade Execution

This notebook retrieves signals from the Firebase `signals` queue, calculates position sizes (ATR/Kelly), and updates the global `trades` state in the cloud.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas python-dotenv


In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase & Secrets)
import json, os, firebase_admin, numpy as np, pandas as pd, yfinance as yf
from firebase_admin import credentials, firestore
from datetime import datetime

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")
            except: tg = None
            return {
                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
                "telegram": tg
            }
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "telegram": os.getenv("TELEGRAM_TOKEN")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
        
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        # Wrap in payload to allow lists
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def get_history(self, limit=5):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        history = [doc.to_dict() for doc in docs]
        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        tables = pd.read_html('https://id.wikipedia.org/wiki/LQ45')
        for df in tables:
            if 'Kode' in df.columns:
                return (df['Kode'] + '.JK').tolist()
            elif 'Ticker' in df.columns:
                return (df['Ticker'] + '.JK').tolist()
    except:
        pass
    return fallback


In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Risk & Execution)
class RiskManager:
    def calculate_size(self, price, conviction, total_equity=100000000):
        # Simple ATR-like sizing demo
        return int((total_equity * 0.02 * conviction) / price)

class TradingAgent:
    def __init__(self, fb):
        self.fb = fb
        self.risk = RiskManager()

    def execute_signals(self, signals):
        for ticker, data in signals.items():
            print(f"🚀 Executing trade for {ticker}...")
            shares = self.risk.calculate_size(data['price'], data['conviction'])
            if shares > 0:
                trade_data = {
                    'ticker': ticker,
                    'shares': shares,
                    'entry_price': data['price'],
                    'entry_date': datetime.now().isoformat(),
                    'conviction': data['conviction'],
                    'status': 'OPEN'
                }
                self.fb.insert_trade(trade_data)
                self.fb.log_event("EXECUTION", f"Opened {ticker} @ {data['price']} ({shares} shares)")

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_execution():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    # 1. Pull signals queue
    signal_batches = fb.get_and_clear_queue("signals")
    if not signal_batches: print("📭 Queue empty."); return
    
    all_signals = {}
    for batch in signal_batches: all_signals.update(batch)

    # 2. Execute
    agent = TradingAgent(fb)
    agent.execute_signals(all_signals)
    print("✅ Execution complete.")

run_execution()